# FOV analysis scheduler

Run **during acquisition**, in its own JupyterLab tab or kernel. Analyses
every new image file continuously, FOVs in parallel, and writes:
- thumbnails (PNG) of each frame
- per-frame intensity stats (CSV)
- intensity histograms (`.npz`)

A zero-byte sentinel marks each finished FOV, for `round_scheduler.ipynb`.

**`ANALYSIS_MODE`:**
* `"same_drive"` -- analyse from `data/` on the acquisition drive. Simplest;
  also the one to use when `data/` is on a NAS.
* `"mirror_drive"` -- during fluidics, mirror `data/` to a second drive
  (`ANALYSIS_SOURCE_DIR`) and analyse from there, so analysis never competes
  with the microscope's writes.

**Parallelism:** `N_ANALYSIS_WORKERS` processes, one FOV each (default:
usable CPUs - 2). Each holds one image stack (~200 MB).

The optional NAS transfer (`TRANSFER_DEST`) runs only during fluidics.

## 1 — Setup

In [ ]:
import os
import sys
import logging
from pathlib import Path

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/measure_stats/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress        import ProgressTracker
from MERci.state           import ExperimentStateMonitor
from MERci.scheduler       import FOVScheduler, RoundScheduler
from MERci.plots.experiment_plots   import display_mosaic

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Experiment parameters

Edit the cells below to match your experiment.

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)

# ── Fluidics type ──────────────────────────────────────────────────────
# "adaptor"  →  t_max = 100 min  (adaptor-based fluidics)
# "direct"   →  t_max =  50 min  (direct-readout fluidics)
FLUIDICS_TYPE = "adaptor"

# ── Image file format (must match what HAL writes) ─────────────────────
IMAGE_SUFFIX = ".zarr"   # options: ".zarr", ".dax", ".tiff"

# ── Analysis mode ──────────────────────────────────────────────────────
# "same_drive"  : analyse from data/ (also for data/ on a NAS)
# "mirror_drive": mirror data/ to a 2nd drive during fluidics, analyse from there
ANALYSIS_MODE       = "same_drive"
# mirror_drive only: folder on a second drive, e.g. r"E:\merci_mirror\LT027\data"
ANALYSIS_SOURCE_DIR = None

# ── Parallelism ────────────────────────────────────────────────────────
# FOV worker processes. None = usable CPUs - 2; 1 = serial, in-process (easiest to debug).
N_ANALYSIS_WORKERS = None

# ── FOV subset (optional) ──────────────────────────────────────────────
# FOV ids to process, e.g. [0, 1, 2, 3]; None = all
FOV_SUBSET = None

# ── Thumbnail frames (optional) ────────────────────────────────────────
# Frame indices to thumbnail, e.g. [2, 3, 4]; None = all
THUMBNAIL_FRAMES = None

# ── Data transfer (optional, independent of ANALYSIS_MODE) ─────────────
# Network path to copy finished rounds to during fluidics, when at least
# TRANSFER_MIN_TIME seconds remain. None = no transfer.
TRANSFER_DEST     = None   # e.g. r"\\NAS\experiments\LT027"
TRANSFER_MIN_TIME = 600    # seconds (10 min)

print(f"Sample name    : {SAMPLE_NAME}")
print(f"Positions tag  : {POSITIONS_TAG}")
print(f"Fluidics type  : {FLUIDICS_TYPE}")
print(f"Image suffix   : {IMAGE_SUFFIX}")
print(f"Analysis mode  : {ANALYSIS_MODE}")
print(f"Analysis source: {ANALYSIS_SOURCE_DIR}")
print(f"Workers        : {N_ANALYSIS_WORKERS}")
print(f"FOV subset     : {FOV_SUBSET}")
print(f"Thumb frames   : {THUMBNAIL_FRAMES}")
print(f"Transfer dest  : {TRANSFER_DEST}")

In [ ]:
config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt       = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    fluidics_type       = FLUIDICS_TYPE,
    image_suffix        = IMAGE_SUFFIX,
    analysis_mode       = ANALYSIS_MODE,
    analysis_source_dir = ANALYSIS_SOURCE_DIR,
    n_analysis_workers  = N_ANALYSIS_WORKERS,
    fov_subset          = FOV_SUBSET,
    thumbnail_frames    = THUMBNAIL_FRAMES,
    transfer_dest       = TRANSFER_DEST,
    transfer_min_time   = TRANSFER_MIN_TIME,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)
monitor = ExperimentStateMonitor(config)

print(f"Rounds         : {meta.n_rounds}")
print(f"FOVs           : {meta.n_fovs}")
print(f"Analysis mode  : {config.analysis_mode}")
print(f"Analysis source: {config.analysis_data_dir}")
print(f"Workers        : {config.resolved_n_workers}")
print(f"Analysis dir   : {config.analysis_dir}")
if config.transfer_dest:
    print(f"Transfer dest  : {config.transfer_dest}  (min time: {config.transfer_min_time:.0f} s)")

## 3 — Check current progress

Run this cell at any time to see how many FOVs and rounds have been processed.

In [ ]:
summary = tracker.summary(meta)
print(f"FOVs  done : {summary['files_fov_done']} / {summary['files_total']}")
print(f"Rounds done: {summary['rounds_done']} / {summary['rounds_total']}")

## 4 — FOV scheduler

**Run this cell and leave it running.** Each tick it refreshes the mirror
(mirror mode, while the microscope is idle), analyses pending files in
parallel, and writes each finished FOV's sentinel.

Interrupt the kernel (`■`) to stop.

> **Windows:** if worker startup fails, set `N_ANALYSIS_WORKERS = 1`.

In [ ]:
def show_phase(phase):
    from IPython.display import clear_output
    clear_output(wait=True)
    tsi = f"{phase.time_since_imaging:.0f} s" if phase.time_since_imaging is not None else "n/a"
    print(f"Phase         : {phase.phase_str}   (analysis runs continuously)")
    print(f"Time since img: {tsi}")
    print(f"Mode / workers: {config.analysis_mode} / {config.resolved_n_workers}")
    smry = tracker.summary(meta)
    print(f"FOVs done     : {smry['files_fov_done']} / {smry['files_total']}")

FOVScheduler(config, meta, tracker, monitor).run_loop(on_phase_update=show_phase)